# Dark Kinase Multi-Class Classifier

Fine-tunes PubMedBERT to classify PTM text descriptions into 4 classes:

| Class | Label | Criteria |
|-------|-------|----------|
| 0 | Direct phosphorylation (light kinase) | Has phospho prefix + amino acid + specific residue location |
| 1 | Potential dark kinase | Has any 2 of the 3 criteria |
| 2 | Dark kinase | Has only 1 of the 3 criteria (vague phosphorylation mention) |
| 3 | Not a phosphorylation event | None of the 3 criteria present |

In [ ]:
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from datasets import Dataset
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from torch.nn import CrossEntropyLoss
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

warnings.filterwarnings("ignore")
%config InlineBackend.figure_format = 'retina'


## 1. Load data and apply rule-based labeling

In [ ]:
df = pd.read_csv("dark_kinome_ptms.csv")
print("Raw dataset shape:", df.shape)
print("\nOriginal output_class distribution (1=bait, 2=prey):")
print(df["output_class"].value_counts().sort_index())


# --- Rule-based labeling heuristics ---
# Three criteria for phosphorylation evidence:
#   1. Phospho prefix  (e.g. "phosphorylated", "phosphorylation")
#   2. Amino acid type  (Tyr / Ser / Thr or full names)
#   3. Specific residue location  (e.g. Tyr-105, Ser-750, S750)


def has_phospho_prefix(text: str) -> bool:
    return bool(re.search(r"(?i)phospho", text))


def has_amino_acid(text: str) -> bool:
    return bool(re.search(r"(?i)\b(Tyr|Ser|Thr|Tyrosine|Serine|Threonine)\b", text))


def has_residue_location(text: str) -> bool:
    return bool(re.search(r"(?i)\b(Tyr|Ser|Thr|Y|S|T)-?\d+\b", text))


def assign_label(text: str) -> int:
    """Assign 4-class label based on phosphorylation evidence criteria."""
    score = (
        int(has_phospho_prefix(text))
        + int(has_amino_acid(text))
        + int(has_residue_location(text))
    )
    if score == 3:
        return 0  # Direct phosphorylation (light kinase)
    elif score == 2:
        return 1  # Potential dark kinase
    elif score == 1:
        return 2  # Dark kinase
    else:
        return 3  # Not a phosphorylation event


df["label"] = df["ptm_text"].apply(assign_label)

CLASS_NAMES = [
    "Direct Phosphorylation",
    "Potential Dark Kinase",
    "Dark Kinase",
    "Not Phosphorylation",
]

print("\nMulti-class label distribution:")
for i, name in enumerate(CLASS_NAMES):
    count = (df["label"] == i).sum()
    print(f"  {i} — {name}: {count}")

print("\nSample entries per class:")
for i, name in enumerate(CLASS_NAMES):
    print(f"\n--- Class {i}: {name} ---")
    samples = df[df["label"] == i]["ptm_text"].head(2)
    for s in samples:
        print(f"  {s[:120]}")


## 2. Train / Validation / Test split

In [ ]:
# Keep only the columns needed for modeling
model_df = df[["ptm_text", "label"]].copy()

# 80 / 10 / 10 split, stratified by label
train_val_df, test_df = train_test_split(
    model_df,
    test_size=0.10,
    random_state=42,
    stratify=model_df["label"],
)

train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.111,  # 0.111 of 90% ≈ 10% of total
    random_state=42,
    stratify=train_val_df["label"],
)

print(f"Train: {len(train_df)}  |  Val: {len(val_df)}  |  Test: {len(test_df)}")
print("\nTrain label distribution:")
for i, name in enumerate(CLASS_NAMES):
    count = (train_df["label"] == i).sum()
    print(f"  {i} — {name}: {count}")


## 3. Tokenization with PubMedBERT

In [ ]:
MODEL_NAME = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Quick sanity check
sample = model_df["ptm_text"].iloc[0]
tokens = tokenizer(sample, truncation=True, max_length=512)
print("Sample text:", sample[:80], "...")
print("Number of tokens:", len(tokens["input_ids"]))


In [ ]:
def tokenize(examples):
    return tokenizer(
        examples["ptm_text"],
        truncation=True,
        padding="max_length",
        max_length=512,
    )


# Convert to HuggingFace Datasets
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

# Tokenize
train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

# Rename label column for HuggingFace Trainer
train_dataset = train_dataset.rename_column("label", "labels")
val_dataset = val_dataset.rename_column("label", "labels")
test_dataset = test_dataset.rename_column("label", "labels")

train_dataset.set_format("torch")
val_dataset.set_format("torch")
test_dataset.set_format("torch")

print("Tokenization complete.")
print("Train dataset features:", train_dataset.features)


## 4. Class weights and weighted trainer

In [ ]:
# Inverse-frequency class weights: weight_i = N_total / (N_classes * N_i)
NUM_CLASSES = 4
total = len(train_df)
class_counts = train_df["label"].value_counts().sort_index()

weights = []
for i in range(NUM_CLASSES):
    n_i = class_counts.get(i, 1)
    w = total / (NUM_CLASSES * n_i)
    weights.append(w)
    print(f"Class {i} ({CLASS_NAMES[i]}): count={n_i}, weight={w:.3f}")

class_weights = torch.tensor(weights, dtype=torch.float)


class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = CrossEntropyLoss(weight=class_weights.to(logits.device))
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1_macro": f1_score(labels, predictions, average="macro"),
        "f1_weighted": f1_score(labels, predictions, average="weighted"),
        "precision_macro": precision_score(labels, predictions, average="macro"),
        "recall_macro": recall_score(labels, predictions, average="macro"),
    }


## 5. Load model (4-class head)

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CLASSES,
)

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = model.to(device)

print("Model loaded and moved to:", device)
print("Total parameters:", f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f} M")
print("Classification head output size:", model.classifier.out_features)


## 6. Training

In [ ]:
training_args = TrainingArguments(
    output_dir="./dkinase_multiclass_model",
    num_train_epochs=8,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_dir="./logs_multiclass",
    logging_steps=10,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

print("Starting training...")
trainer.train()
print("Training complete.")


## 7. Evaluation on held-out test set

In [ ]:
print("Evaluating on held-out test set...\n")
predictions = trainer.predict(test_dataset)
preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

print(classification_report(labels, preds, target_names=CLASS_NAMES))

# Confusion matrix
cm = confusion_matrix(labels, preds)
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix — Test Set (4-Class)")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
## 0 shot with PubMedBERT

# Bio-Mistral
# Stratified test-train split



## 8. Save model

In [ ]:
trainer.save_model("./dkinase_multiclass_final")
tokenizer.save_pretrained("./dkinase_multiclass_final")
print("Model saved to ./dkinase_multiclass_final")


## 9. Interactive prediction

In [ ]:
def predict(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=True,
    ).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1).cpu().numpy()[0]
    pred = np.argmax(probs)
    print(f"Prediction: {CLASS_NAMES[pred]}")
    print(f"Confidence: {probs[pred] * 100:.1f}%")
    for i, name in enumerate(CLASS_NAMES):
        print(f"  P({name}): {probs[i] * 100:.1f}%")
    print()


# Test examples spanning all 4 classes
print("=== Direct phosphorylation (all 3 criteria) ===")
predict("Phosphorylated by TBK1 at Ser-172, which activates the kinase")

print("=== Potential dark kinase (2 criteria: phospho + amino acid, no residue) ===")
predict("Phosphorylated on tyrosine residues")

print("=== Dark kinase (1 criterion: just phospho prefix) ===")
predict("Phosphorylated")

print("=== Not phosphorylation ===")
predict("Ubiquitinated by STUB1/CHIP, leading to proteasomal degradation")
